In [1]:
import numpy as np
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import deeplake 
import torch
from PIL import Image
from torchvision import transforms


class ToPILCheck:
    def __call__(self, img):
        if isinstance(img, Image.Image):
            return img

        else:
            return transforms.ToPILImage()(img)
        

class RGBCheck:
    def __call__(self, img: Image):
        return img.convert("RGB")


        
def resize(img: np.ndarray | Image.Image) -> torch.Tensor:
    
    img_transform = transforms.Compose([
        ToPILCheck(),
        RGBCheck(),
        transforms.Resize((224, 224)),
        transforms.ToTensor()])
        

    transformed = img_transform(img)

    return transformed

'''transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))'''


class patches_loader(Dataset):
    '''
    Class to handle patches stored as deeplake objects
    '''
    def __init__(self, train_or_test, WSI_id, scanner, to_torch=False, emb=True):
        self.train_or_test = train_or_test
        self.WSI_id = WSI_id
        self.scanner = scanner
        directory = f"/home/leolr-int/nfs/data/data/patched/dim_256/{train_or_test}"
        #here we consider only Subset3
        WSI = f'Subset3_{train_or_test}_{WSI_id}_{scanner}'
        self.patches = deeplake.open_read_only(f'{directory}/{WSI}')
        self.to_torch = to_torch
        self.emb = emb

    def summary(self):
        return self.patches.summary()
    
    # to specify a label and then access the columns of the deeplake dataset
    def __getitem__(self, idx):
        if self.to_torch:
            patch = self.patches[idx]
            
            img = patch["patch"].copy()
    
            '''if preprocess_fn is not None:
                img = preprocess_fn(img)
        
            if apply_augmentation:
                img = augment_fn(img)'''
        
            img = resize(img)
            
            #img = torch.tensor(img)
            label = torch.tensor(patch["label"], dtype=torch.long)
            area = torch.tensor(patch["area"], dtype=torch.long)
            x = torch.tensor(patch["x"], dtype=torch.long)
            y = torch.tensor(patch["y"], dtype=torch.long)
            w = torch.tensor(patch["w"], dtype=torch.long)
            h = torch.tensor(patch["h"], dtype=torch.long)
            
            metadata = {
                "area": area,
                "x": x,
                "y": y,
                "w": w,
                "h": h,
                }
            
            dic = {'img':img, 'label':label, 'metadata': metadata}

            if self.emb:
                # connection to embeddings
                directory = '/home/leolr-int/nfs/transformed_data/my_embeddings'
                WSI = f'Subset3_{self.train_or_test}_{self.WSI_id}_{self.scanner}'
                embedding_ds = deeplake.open_read_only(f'{directory}/{WSI}')
                embedding = embedding_ds[idx]['embedding']
                embedding = torch.tensor(embedding, dtype=torch.float)
                dic['embedding'] = embedding
            
            return dic
            
        else: 
            #deeplake object
            return self.patches[idx]
            # Example: patches[idx]['label']

    def __len__(self):
        return len(self.patches)
    
    def display(self, idx): 
        fig, axes = plt.subplots(figsize=(4, 4))
        axes.imshow(self.patches[idx]["patch"])
        plt.show()

    
    def to_embedding(self, idx=None):
        # connection to embeddings
        directory = '/home/leolr-int/nfs/transformed_data/my_embeddings'
        WSI = f'Subset3_{self.train_or_test}_{self.WSI_id}_{self.scanner}'
        embedding_ds = deeplake.open_read_only(f'{directory}/{WSI}')
        if idx == None:
            embeddings_np = np.array(embedding_ds['embedding'])  # stack into 1 array
            embeddings_tensor = torch.from_numpy(embeddings_np).float()
            return embeddings_tensor 
        else:
            embedding = embedding_ds[idx]['embedding']
            embedding = torch.tensor(embedding, dtype=torch.float)
        return embedding
      




In [2]:
#test
'''
idx=1500
file_test = patches_loader('Train', 1, 'Leica', to_torch=True)
file_test.display(idx)
file_test[idx]['img'].shape
#file_test.summary()
#print(file_test.to_embedding())
#file_test[idx]['embedding'] #use this form only when to_torch = True
#file_test.to_embedding(idx)'''


"\nidx=1500\nfile_test = patches_loader('Train', 1, 'Leica', to_torch=True)\nfile_test.display(idx)\nfile_test[idx]['img'].shape\n#file_test.summary()\n#print(file_test.to_embedding())\n#file_test[idx]['embedding'] #use this form only when to_torch = True\n#file_test.to_embedding(idx)"

In [3]:
class multi_WSI_loader(Dataset):
    '''
    Class to handle several WSI from different scanners
    Used for training a neural network
    '''

    def __init__(self, WSI_ids, scanner, train_or_test='Train'):
        self.train_or_test = train_or_test
        self.WSI_ids = WSI_ids
        self.scanner = scanner
        
        # dictionary to store all the patches_loader objects
        self.datasets = []
        
        for WSI_id in WSI_ids: 
            ds = patches_loader(train_or_test, WSI_id, scanner, to_torch=True)
            _ = len(ds)
            self.datasets.append(ds)

        # index mapping
        self.index_map = []
        for ds_idx, ds in enumerate(self.datasets):
            for i in range(len(ds)):
                self.index_map.append((ds_idx, i))


    # define indexing so that the dataloader can access data    
    def __getitem__(self, idx):
        ds_idx, patch_idx = self.index_map[idx]
        return self.datasets[ds_idx][patch_idx] #which is a patches_loader object
    
    def __len__(self):
        return len(self.index_map)


def make_multi_WSI_loader(WSI_ids, scanners, train_or_test, batch_size):
    datasets = []
    
    for scanner in scanners:
        dataset = multi_WSI_loader(WSI_ids, scanner, train_or_test)
        datasets.append(dataset)
    
    datasets = ConcatDataset(datasets)
    loader = DataLoader(datasets, batch_size=64, shuffle=True, num_workers=6)

    return loader
    

In [4]:
Test = False

if Test: 
    train_scanners = ['Akoya', 'Leica', 'KFBio']
    WSI_ids = [1,2]
    target_scanner = ['Akoya'] if 'Akoya' in train_scanners else random.choice(train_scanners)
    train_scanners.remove(target_scanner[0])
    source_scanner = train_scanners
    
    print(target_scanner)
    print(source_scanner)
    
    target_dataset = multi_WSI_loader(WSI_ids, target_scanner, train_or_test='Train')
    source_dataset = multi_WSI_loader(WSI_ids, source_scanner, train_or_test='Train')
    
    
    # DataLoaders
    batch_size = 16
    train_loader_source = DataLoader(source_dataset, batch_size=batch_size, shuffle=True, num_workers=4, drop_last=True)
    train_loader_target = DataLoader(target_dataset, batch_size=batch_size, shuffle=True, num_workers=4, drop_last=True)
    
    
    for batch in train_loader_source:
        images = batch['img']
        labels = batch['label']
        
        print("Source batch - Images shape:", images.shape, "Labels:", labels)
        break
    
    for batch in train_loader_target:
        images = batch['img']
        labels = batch['label']
        embeddings = batch['embedding']
        
        print("Target batch - Images shape:", images.shape, "Labels:", labels, "Emb:", embeddings)
        break

    



## Test with network handler


In [5]:
import torch
import torch.nn as nn
import random

class Neural_Network(nn.Module):
    """
    Initialises an Artificial Neural Network with the foundation encoder Gigapath 
    and 2 layers for classification (one to create embeddings, one to classify)

    """

    def __init__(self, BASE_MODEL_DIR, freeze_encoder: bool = True, OT: bool = False, num_classes: int = 5):
        super().__init__() #super constructor for ANN in PyTorch

        self.freeze_encoder = freeze_encoder
        self.OT = OT #maybe not useful here
        
        # Define encoder
        encoder_name = 'gigapath'
        encoder_dir = os.path.join(BASE_MODEL_DIR, "pre_trained_weights")
        encoder_path = os.path.join(encoder_dir, f"{encoder_name}.pth")
        encoder = torch.load(encoder_path, map_location=torch.device("cpu"), weights_only=False)
        self.encoder = encoder

        if self.freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False

        # Define bottle neck / embeddings
        # fixed parameter value for Gigapath
        in_dim = 1536
        self.bottle_neck = nn.Sequential(
            nn.Linear(in_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(), #do not forget ReLU
            nn.Dropout(p=0.5))

        # Define classification head
        out_dim = num_classes
        self.head = nn.Linear(1024, out_dim)

        # Define sequential architecture
        def forward(self, x): 
            if self.freeze_encoder: 
                with torch.no_grad():
                    encoded = self.encoder(x)
                embedding = self.bottle_neck(encoded)
            else:
                embedding = self.bottle_neck(self.encoder(x))
            logits = self.head(embedding)
            return logits

        


In [6]:
# Training graph

def train_plot(training_stats, cm):
    fig, axes = plt.subplots(2,2, figsize=(10,10))
    # loss curves
    axes[0,0].plot(training_stats['epoch_loss_train'], label='Training loss', color='blue')
    axes[0,0].plot(training_stats['epoch_loss_val'], label='Validation loss', color='green')
    axes[0,0].set_xlabel('Epoch')
    axes[0,0].set_ylabel('Loss')
    axes[0,0].legend()
    axes[0,0].set_title('Loss per epoch')

    # accuracy curves
    axes[0,1].plot(training_stats['epoch_balanced_accuracy_train'], label='Training accuracy', color='blue')
    axes[0,1].plot(training_stats['epoch_balanced_accuracy_val'], label='Validation accuracy', color='green')
    axes[0,1].set_xlabel('Epoch')
    axes[0,1].set_ylabel('Accuracy')
    axes[0,1].legend()
    axes[0,1].set_title('Accuracy per epoch')

    # time per epoch
    axes[1,0].plot(training_stats['time'], color='black')
    axes[1,0].set_xlabel('Epoch')
    axes[1,0].set_ylabel('Time')
    axes[1,0].set_title('Time for each epoch')

    # confusion matrix
    label_name = ['Stroma', 'Normal', 'G3', 'G4', 'G5']
    display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_name)
    display.plot(ax=axes[1,1], cmap='PuRd')
    axes[1,1].set_title('Confusion matrix for validation')

    plt.suptitle('Training statistics')
    plt.savefig('training_stats.png')
    plt.close()

In [7]:
test = False
if test:

    # save checkpoint
    def save_checkpoint(
        save_dir: str,
        model: nn.Module, 
        optimizer: torch.optim.Optimizer,
        scheduler: torch.optim.lr_scheduler,
        scaler: torch.amp.GradScaler,
        epoch: int,
        balanced_accuracy: torch.Tensor | float,
        loss: torch.Tensor | float,
        min_val_loss: torch.Tensor | float,
        max_val_accuracy: torch.Tensor | float
        ):
    
        training_state = {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "epoch": epoch,
            "balanced_accuracy": balanced_accuracy,
            "loss": loss,
            "min_val_loss": min_val_loss,
            "max_val_accuracy": max_val_accuracy
        }
    
        torch.save(training_state, os.path.join(save_dir, "checkpoint.pth"))
    
    
    trainable_params = list(filter(lambda p: p.requires_grad, model.parameters()))
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(trainable_params, lr=args["learning_rate"], weight_decay=args["weight_decay"])
    scheduler = CosineAnnealingLR(optimizer, args["epochs"], eta_min=args["eta_min"])
    
    min_val_loss, max_val_accuracy = inf, -inf
    
    for epoch in range(1, args["epochs"] + 1):
        print("\n" + f"Epoch [{epoch}/{args['epochs']}]".center(BORDER_WIDTH))
        print(f"{'=' * BORDER_WIDTH}\n")
    
        writer.add_scalar("Learning Rate", scheduler.optimizer.param_groups[0]["lr"], epoch)
    
        train_loss, train_accuracy = network_handler.train_epoch(train_loader)
        log_metrics(writer=writer, loss=train_loss, prefix="Train", epoch=epoch, performance=train_accuracy)
    
        print(f"{'-' * BORDER_WIDTH}\n")
    
        val_loss, val_accuracy = network_handler.validate_epoch(val_loader)
        log_metrics(writer=writer, loss=val_loss, prefix="Validation", epoch=epoch, performance=val_accuracy)
    
        if val_loss < min_val_loss:
            torch.save(model.fc.state_dict(), os.path.join(model_dir, f"lowest_loss.pth"))
            min_val_loss = val_loss
            print("New minimum loss — model saved.")
        
        if val_accuracy > max_val_accuracy:
            torch.save(model.fc.state_dict(), os.path.join(model_dir, f"highest_balanced_accuracy.pth"))
            max_val_accuracy = val_accuracy
            print("New maximum balanced accuracy — model saved.")
    
        scheduler.step()
    
        print(f"{'=' * BORDER_WIDTH}\n")
    
    print("Run Summary:")
    print(f"Min Loss: {min_val_loss:.4f} | Max Balanced Accuracy: {max_val_accuracy:.4f}\n")


In [8]:
def save_checkpoint(
    save_dir,
    custom_name,
    model,
    optimizer,
    scheduler,
    epoch,
    balanced_accuracy,
    loss,
    min_loss_val,
    max_accuracy_val):

    os.makedirs(f'{save_dir}/{custom_name}', exist_ok=True)

    training_state = {
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'epoch': epoch,
        'balanced_accuracy': balanced_accuracy,
        'loss': loss,
        'min_loss_val': min_loss_val,
        'max_accuracy_val': max_accuracy_val
    }

    torch.save(training_state, os.path.join(save_dir, custom_name, 'checkpoint.pth'))

# i have to give a custom_name

def end_epoch(
    save_dir,
    custom_name,
    model,
    optimizer,
    scheduler,
    epoch,
    epoch_loss_train,
    epoch_balanced_accuracy_train,
    epoch_loss_val,
    epoch_balanced_accuracy_val,
    min_loss_val,
    max_accuracy_val
):
    save_checkpoint(
        save_dir=save_dir,
        custom_name=custom_name,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        epoch=epoch,
        balanced_accuracy=epoch_balanced_accuracy_val,
        loss=epoch_loss_val,
        min_loss_val =min_loss_val,
        max_accuracy_val=max_accuracy_val,
    )

    if epoch_loss_val < min_loss_val:
        torch.save(model.state_dict(), os.path.join(save_dir, custom_name, 'lowest_loss.pth'))
        min_loss_val = epoch_loss_val
        print(f'Epoch {epoch}: new minimum for val loss = {min_loss_val}')
    if epoch_balanced_accuracy_val > max_accuracy_val:
        torch.save(model.state_dict(), os.path.join(save_dir, custom_name, 'max_accuracy.pth'))
        max_accuracy_val = epoch_balanced_accuracy_val
        print(f'Epoch {epoch}: new maximum for val accuracy = {max_accuracy_val}')
                   

        return min_loss_val, max_accuracy_val



In [9]:
import os
from typing import (
    Tuple, 
    Literal
)
import random
import torch
import numpy as np
import torch.nn as nn
from tqdm import tqdm
import torch.nn.functional as F
from torch.amp import GradScaler
from torch.utils.data import DataLoader
import deeplake
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from geomloss import SamplesLoss
import time
import pandas as pd
from torch.utils.data import ConcatDataset
from torch.optim.lr_scheduler import CosineAnnealingLR

# Ensuring reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(42)
random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False 

# Define global variables
#metrics_train = {'running_loss':0, 'predictions':[], 'labels':[]}
#metrics_val = {'running_loss':0, 'predictions':[], 'labels':[]}


# Defining OT-based loss function
loss_geom = SamplesLoss('sinkhorn', p=2, blur=0.1, scaling=0.95, verbose=False)
Lambda = 0.1 # strength of OT (0.1 is the value of the article)

class NetworkHandler:
    '''
    A class to handle training, inference and prediction
    '''

    def __init__(self, precision = 'mixed', freeze_encoder = True, embedding_mode = False, display = False):
        self.precision = precision
        self.freeze_encoder = freeze_encoder
        self.embedding_mode = embedding_mode
        self.display = display

        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        BASE_MODEL_DIR = '/home/leolr-int/AGGCPerturbations/model_weights'
        self.model = Neural_Network(BASE_MODEL_DIR)
        self.model = self.model.to(self.device)

        self.use_amp = precision == 'mixed' and self.device == 'cuda'
        self.grad_scaler = GradScaler(enabled=self.use_amp)

    
    def training_no_OT(self, scanners_train, batch_size, num_epochs):
        # here we train only using cross entropy
        training_stats = []
        min_loss_val, max_accuracy_val = float("inf"), -float("inf")

        trainable_params = list(filter(lambda p: p.requires_grad, self.model.parameters()))
        optimizer = torch.optim.AdamW(trainable_params, lr=0.03, weight_decay=0.001)
        scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6) #to be changed maybe
        
        for epoch in range(0, num_epochs):

            metrics_train = {'running_loss': 0, 'predictions': [], 'labels': []}
            metrics_val   = {'running_loss': 0, 'predictions': [], 'labels': []}
            
            start = time.time()
            # 1st part: training for one epoch
    
            '''optimizer = torch.optim.SGD(self.model.parameters(), lr=0.03, momentum=0.9, weight_decay=0.001)
            torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.99)'''
    
            #data loading (equal number of patchs per scanner)
            loader_train = make_multi_WSI_loader(WSI_ids_train, scanners_train, train_or_test='Train', batch_size=batch_size)
            
            
            self.model.train()
    
            #deactivate the encoder training if needed
            if self.freeze_encoder or self.embedding_mode: 
                self.model.encoder.eval()
            
            pbar = tqdm(loader_train, desc='Training with Cross-Entropy in progress')
    
            for batch in pbar:
                patch = batch['embedding'] if self.embedding_mode else batch['img'] #correct axes for Gigapath
                patch = patch.to(self.device)
                label = batch['label'].to(self.device)
    
                with torch.autocast(device_type = self.device, dtype = torch.float16, enabled = self.use_amp):
                    #if embeddings from gigapath are already computed, we can speed up training
                    logits = self.model.bottle_neck(patch) if self.embedding_mode else self.model(patch)
                    loss = nn.CrossEntropyLoss()(logits, label) #careful about syntax
                
                self.grad_scaler.scale(loss).backward()
                self.grad_scaler.step(optimizer)
                scheduler.step()
                self.grad_scaler.update()
                optimizer.zero_grad()
    
                confidence = F.softmax(logits, dim=1)
                pred = torch.argmax(confidence, dim=1)
                
                #performance metrics
                metrics_train['running_loss'] += loss.detach().cpu().item()
                metrics_train['predictions'].extend(pred.cpu().numpy())
                metrics_train['labels'].extend(label.cpu().numpy())
    
                pbar.set_postfix({'step_loss': loss.detach().cpu().item()})
            
            epoch_loss_train = metrics_train['running_loss'] / len(loader_train)
            epoch_balanced_accuracy_train = balanced_accuracy_score(metrics_train['labels'], metrics_train['predictions'])
            
    
            # 2nd part: validation for one epoch 
            with torch.no_grad():
                self.model.eval()
                
                #we still work with the Train folder
                loader_val = make_multi_WSI_loader(WSI_ids_val, scanners_train, train_or_test='Train', batch_size=batch_size)
                
                
                pbar = tqdm(loader_val, desc='Validation with Cross-Entropy in progress')
                for batch in pbar:
                    patch = batch['embedding'] if self.embedding_mode else batch['img']
                    patch = patch.to(self.device)
                    label = batch['label'].to(self.device)
        
                    with torch.autocast(device_type=self.device, dtype=torch.float16, enabled=self.use_amp):
                        logits = self.model.bottle_neck(patch) if self.embedding_mode else self.model(patch)
                        loss = nn.CrossEntropyLoss()(logits, label)
                    
                    confidence = F.softmax(logits, dim=1)
                    pred = torch.argmax(confidence, dim=1)
        
                    metrics_val["running_loss"] += loss.detach().cpu().item()
                    metrics_val["predictions"].extend(pred.cpu().numpy())
                    metrics_val["labels"].extend(label.cpu().numpy())
        
                    pbar.set_postfix({"step_loss": loss.detach().cpu().item()})
        
                epoch_loss_val = metrics_val["running_loss"] / len(loader_val)
                epoch_balanced_accuracy_val = balanced_accuracy_score(metrics_val["labels"], metrics_val["predictions"])
                
                cm = confusion_matrix(metrics_val["predictions"], metrics_val["labels"], labels=[0, 1, 2, 3, 4], normalize='true')
    
                end = time.time()
    
                dic = {'epoch_loss_train': epoch_loss_train, 
                    'epoch_balanced_accuracy_train': epoch_balanced_accuracy_train, 
                    'epoch_loss_val': epoch_loss_val, 
                    'epoch_balanced_accuracy_val': epoch_balanced_accuracy_val, 
                    'time': end - start,
                    'cm':cm}
    
                training_stats.append(dic)

                min_val_loss, max_val_accuracy = end_epoch(
                                                    save_dir,
                                                    custom_name,
                                                    self.model,
                                                    optimizer,
                                                    scheduler,
                                                    epoch,
                                                    epoch_loss_train,
                                                    epoch_balanced_accuracy_train,
                                                    epoch_loss_val,
                                                    epoch_balanced_accuracy_val,
                                                    min_loss_val,
                                                    max_accuracy_val)
                
                train_plot(pd.DataFrame(training_stats), cm)
            
        return epoch_loss_val, epoch_balanced_accuracy_val

    @torch.no_grad()
    def extract_embeddings(self, scanners, WSI_ids, train_or_test, batch_size):
        # creates the deeplake database for embeddings
        # structure: one deeplake dataset per WSI, embeddings, scanner, WSI_id, Train or Test

        root_dir = '/home/leolr-int/nfs/transformed_data/my_embeddings'
        self.model.eval()
        for scanner in scanners:
            for id in WSI_ids:

                path = f'Subset3_{train_or_test}_{id}_{scanner}'
                final_destination = os.path.join(root_dir, path)
                os.makedirs(final_destination, exist_ok=True)
                # creation of the deeplake dataset
                embedding_ds = deeplake.create(final_destination)
                embedding_ds.add_column('embedding', dtype=deeplake.types.Embedding(1536)) 
                embedding_ds.add_column('scanner', dtype=deeplake.types.Text)
                embedding_ds.add_column('WSI_id', deeplake.types.Int32)
                embedding_ds.add_column('train_or_test', dtype=deeplake.types.Text)

                WSI = patches_loader(train_or_test, id, scanner, to_torch=True, emb=False)
                loader = DataLoader(WSI, batch_size=batch_size, shuffle=False, num_workers=4)
                batch_records = []

                for batch in tqdm(loader, desc=f'Extracting {train_or_test}_{id}_{scanner}'):
                    patches = batch['img'].to(self.device).float()

                    with torch.autocast(device_type=self.device, dtype=torch.float16, enabled=self.use_amp):
                        embeddings = self.model.encoder(patches)
                        embeddings = embeddings.detach().cpu().numpy()

                        # accumulate all rows from batch
                        for emb in embeddings:
                            batch_records.append({
                                'embedding':emb,
                                'WSI_id': id, 
                                'scanner': scanner,
                                'train_or_test': train_or_test
                            })

                        # append in large
                        if len(batch_records) >= 1000:
                            embedding_ds.append(batch_records)
                            batch_records.clear()

                    # append what is left
                    if batch_records:
                        embedding_ds.append(batch_records)
                    
                


 


In [10]:
torch.cuda.empty_cache()
scanners_train = ['Akoya', 'Leica'] #add Leica later
train_or_test = 'Train'
WSI_ids_train = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15] #for testing
WSI_ids_val = [16,17] #i have to be sure that all labels are represented in validation data
batch_size = 64
training_stats = []
handler = NetworkHandler(embedding_mode=True)
save_dir = '/home/leolr-int/nfs/transformed_data/weights'
custom_name = 'baseline'

num_epochs = 30
handler.training_no_OT(scanners_train, batch_size, num_epochs)






/home/leolr-int/micromamba/envs/py312-poetry/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Training with Cross-Entropy in progress: 100%|█████████████████████████| 167/167 [00:11<00:00, 14.61it/s, step_loss=4]
/home/leolr-int/micromamba/envs/py312-poetry/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
Validation with Cross-Entropy in progress: 100%|██████████████████████| 94/94 [00:06<00:00, 14.85it/s, step_loss=6.55]


Epoch 0: new minimum for val loss = 6.59716083648357
Epoch 0: new maxim for val accuracy = 0.24296963724307227


Training with Cross-Entropy in progress: 100%|██████████████████████| 167/167 [00:11<00:00, 14.18it/s, step_loss=4.22]
/home/leolr-int/micromamba/envs/py312-poetry/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
Validation with Cross-Entropy in progress: 100%|██████████████████████| 94/94 [00:06<00:00, 14.40it/s, step_loss=6.17]


Epoch 1: new minimum for val loss = 5.952725669170948
Epoch 1: new maxim for val accuracy = 0.2810120879599215


Training with Cross-Entropy in progress: 100%|██████████████████████| 167/167 [00:12<00:00, 13.48it/s, step_loss=3.36]
/home/leolr-int/micromamba/envs/py312-poetry/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
Validation with Cross-Entropy in progress: 100%|██████████████████████| 94/94 [00:06<00:00, 14.35it/s, step_loss=5.56]


Epoch 2: new minimum for val loss = 5.751826144279318
Epoch 2: new maxim for val accuracy = 0.2746612726586379


Training with Cross-Entropy in progress: 100%|██████████████████████| 167/167 [00:12<00:00, 13.08it/s, step_loss=3.88]
/home/leolr-int/micromamba/envs/py312-poetry/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
Validation with Cross-Entropy in progress: 100%|██████████████████████| 94/94 [00:07<00:00, 12.62it/s, step_loss=5.89]


KeyboardInterrupt: 

In [ ]:


'''
